In [6]:
import fitz             
import unicodedata
import re
import pandas as pd

In [2]:
def parse_ficha(block: str) -> dict:
    data = {}
    # No. Cámara
    m = re.search(r"No\. CAMARA:?\s*(\d+)/(\d{4})([CS])", block)
    if m:
        data['num_camara']   = m.group(1)
        data['anio_camara']  = m.group(2)
        data['origen_camara']= 'Camara' if m.group(3)=='C' else 'Senado'
    # No. Senado
    m = re.search(r"No\. SENADO:?\s*(\d+)/(\d{4})([CS])", block)
    if m:
        data['num_senado']   = m.group(1)
        data['anio_senado']  = m.group(2)
        data['origen_senado']= 'Camara' if m.group(3)=='C' else 'Senado'
    # Fecha de radicación
    m = re.search(r"FECHA DE RADICACIÓN:?\s*([0-9]{2}/[0-9]{2}/[0-9]{4})", block)
    if m: data['fecha_radicacion'] = m.group(1)
    # Tipo de proyecto
    m = re.search(r"TIPO DE PROYECTO:?\s*(.+)", block)
    if m: data['tipo_proyecto'] = m.group(1).strip()
    # Seudónimo
    m = re.search(r"SEUD[Nn]IMO:?\s*(.+)", block)
    if m: data['seudonimo'] = m.group(1).strip()
    # Comisión
    m = re.search(r"COMISIÓN:?\s*(.+)", block)
    if m: data['comision'] = m.group(1).strip()
    # Cámara de origen
    m = re.search(r"CÁMARA DE ORIGEN:?\s*(.+)", block)
    if m: data['camara_origen'] = m.group(1).strip()
    # Título
    m = re.search(r"T[ÍI]TULO:?\s*(.+)", block)
    if m: data['titulo'] = m.group(1).strip()
    # Autor(es)
    m = re.search(r"AUTOR(?:ES)?:?\s*(.+)", block)
    if m: data['autores'] = re.sub(r"\s+", " ", m.group(1).strip())

    # Primera vuelta
    prim = re.search(r"PRIMERA VUELTA([\s\S]+?)(?=SEGUNDA VUELTA)", block)
    if prim:
        txt = prim.group(1)
        for lbl in ["Ponentes Primer Debate Camara",
                    "Ponentes Segundo Debate Camara",
                    "Ponentes Primer Debate Senado",
                    "Ponentes Segundo Debate Senado"]:
            m = re.search(fr"{lbl}:?\s*([\s\S]+?)(?=\n[A-Z]|$)", txt)
            if m: data[lbl.lower().replace(" ", "_")] = m.group(1).strip().replace("\n"," ")
        # Publicaciones
        m = re.search(r"PUBLICACIONES GACETA:([\s\S]+?)(?=CAMARA DE REPRESENTANTES:)", txt)
        if m: data['pub_gaceta'] = m.group(1).strip().replace("\n"," ")
        m = re.search(r"CAMARA DE REPRESENTANTES:([\s\S]+?)(?=SENADO DE LA REPUBLICA:)", txt)
        if m: data['pub_camara_rep'] = m.group(1).strip().replace("\n"," ")
        m = re.search(r"SENADO DE LA REPUBLICA:([\s\S]+?)(?=Miembros comisión de conciliación Cámara:)", txt)
        if m: data['pub_senado_rep'] = m.group(1).strip().replace("\n"," ")
        # Miembros comisión conciliación
        m = re.search(r"Miembros comisión de conciliación Cámara:([\s\S]+?)(?=Miembros comisión de conciliación Senado:)", txt)
        if m: data['miembros_conc_cam'] = m.group(1).strip().replace("\n"," ")
        m = re.search(r"Miembros comisión de conciliación Senado:([\s\S]+?)(?=Acto Legislativo:)", txt)
        if m: data['miembros_conc_sen'] = m.group(1).strip().replace("\n"," ")
        # Actos legislativos y observaciones
        acts = re.findall(r"Acto Legislativo: Diario Oficial No\.?\s*(\d+)", txt)
        if acts: data['actos_legislativos'] = ", ".join(acts)
        m = re.search(r"Observaciones:([\s\S]+?)(?=$)", txt)
        if m: data['observaciones'] = m.group(1).strip().replace("\n"," ")

    # Segunda vuelta
    sec = re.search(r"SEGUNDA VUELTA([\s\S]+?)(?=ESTADO ACTUAL)", block)
    if sec:
        txt = sec.group(1)
        for lbl in ["Ponentes Primer Debate Camara",
                    "Ponentes Segundo Debate Camara",
                    "Ponentes Primer Debate Senado",
                    "Ponentes Segundo Debate Senado"]:
            m = re.search(fr"{lbl}:?\s*([\s\S]+?)(?=\n[A-Z]|$)", txt)
            if m: data["2v_"+lbl.lower().replace(" ", "_")] = m.group(1).strip().replace("\n"," ")
        # repetir parsers de PUBLICACIONES, MIEMBROS y ACTOS si es necesario
        m = re.search(r"PUBLICACIONES GACETA:([\s\S]+?)(?=CAMARA DE REPRESENTANTES:)", txt)
        if m: data['2v_pub_gaceta'] = m.group(1).strip().replace("\n"," ")
        m = re.search(r"CAMARA DE REPRESENTANTES:([\s\S]+?)(?=SENADO DE LA REPUBLICA:)", txt)
        if m: data['2v_pub_camara_rep'] = m.group(1).strip().replace("\n"," ")
        m = re.search(r"SENADO DE LA REPUBLICA:([\s\S]+?)(?=Miembros comisión de conciliación Cámara:)", txt)
        if m: data['2v_pub_senado_rep'] = m.group(1).strip().replace("\n"," ")
        m = re.search(r"Miembros comisión de conciliación Cámara:([\s\S]+?)(?=Miembros comisión de conciliación Senado:)", txt)
        if m: data['2v_miembros_conc_cam'] = m.group(1).strip().replace("\n"," ")
        m = re.search(r"Miembros comisión de conciliación Senado:([\s\S]+?)(?=Acto Legislativo:)", txt)
        if m: data['2v_miembros_conc_sen'] = m.group(1).strip().replace("\n"," ")
        acts = re.findall(r"Acto Legislativo: Diario Oficial No\.?\s*(\d+)", txt)
        if acts: data['2v_actos_legislativos'] = ", ".join(acts)
        m = re.search(r"Observaciones:([\s\S]+?)(?=$)", txt)
        if m: data['2v_observaciones'] = m.group(1).strip().replace("\n"," ")

    # Estado actual
    m = re.search(r"ESTADO ACTUAL:([\s\S]+)$", block)
    if m: data['estado_actual'] = m.group(1).strip().replace("\n"," ")

    return data

In [3]:
def main(pdf_path: str, excel_path: str):
    # 1) Leer todo el texto
    with pdfplumber.open(pdf_path) as pdf:
        full_text = "\n".join(p.extract_text() or "" for p in pdf.pages)

    # 2) Separar por cada ficha técnica (empieza con "No. CAMARA")
    bloques = re.split(r"(?=No\. CAMARA)", full_text)
    registros = []
    for b in bloques:
        if "No. CAMARA" in b:
            reg = parse_ficha(b)
            registros.append(reg)

    # 3) DataFrame y Excel
    df = pd.DataFrame(registros)
    df.to_excel(excel_path, index=False)
    print(f"Guardado en {excel_path}")

In [4]:
if __name__ == "__main__":
    import sys
    pdf_path   = r"C:\Users\juans\Documents\proarchitecg\Model-Extract-information\extract\modelos\team\2022 2023 LEGISLATURA_proyectos_ley_actos_Legislativos.pdf"
    excel_path = "fichas_tecnicas.xlsx"
    main(pdf_path, excel_path)

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

Guardado en fichas_tecnicas.xlsx
